# 04.7 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/07-cross-validation/solutions.ipynb)

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

ten = pd.DataFrame({
    "row": range(1, 11),
    "days_since_last": [5, 80, 12, 60, 28, 95, 42, 18, 72, 52],
    "churned":         [0,  1,  0,  1,  0,  1,  1,  0,  0,  1],
})
X10, y10 = ten[["days_since_last"]], ten["churned"]
ten

,row,days_since_last,churned
0,1,5,0
1,2,80,1
2,3,12,0
3,4,60,1
4,5,28,0
5,6,95,1
6,7,42,1
7,8,18,0
8,9,72,0
9,10,52,1


## Exercise 1

Fold A tests rows 1–5 (days 5, 80, 12, 60, 28), trained on rows 6–10 (95/1, 42/1, 18/0, 72/0, 52/1):
5 → 18 → 0 ✓; 80 → 72 → 0 ✗; 12 → 18 → 0 ✓; 60 → 52 → 1 ✓; 28 → 18 → 0 ✓. Accuracy 4/5 = 0.8.

Fold B tests rows 6–10, trained on rows 1–5 (5/0, 80/1, 12/0, 60/1, 28/0):
95 → 80 → 1 ✓; 42 → 28 → 0 ✗; 18 → 12 → 0 ✓; 72 → 80 → 1 ✗; 52 → 60 → 1 ✓. Accuracy 3/5 = 0.6.

Mean 0.7. Fewer folds means each model trains on less data (5 rows instead of 8).

In [2]:
from sklearn.model_selection import KFold, cross_val_score, LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier
print(cross_val_score(KNeighborsClassifier(1), X10, y10, cv=KFold(2)))

[0.8 0.6]


## Exercise 2

In [3]:
from sklearn.model_selection import cross_val_predict
pred = cross_val_predict(KNeighborsClassifier(1), X10, y10, cv=LeaveOneOut())
ten["loo_pred"] = pred
print("LOO accuracy:", (pred == y10).mean())
ten[ten["loo_pred"] != ten["churned"]]

LOO accuracy: 0.8


,row,days_since_last,churned,loo_pred
1,2,80,1,0
8,9,72,0,1


Rows 2 (80 days, churned) and 9 (72 days, stayed) are each other's nearest neighbour and have opposite labels, so whichever one is held out gets the other's label. LOO gives 0.8, the same as 5-fold here, using 10 models instead of 5.

## Exercise 3

In [4]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")                       # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])
features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%
train 2,487 / test 830


In [5]:
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
for k in [3, 5, 10, 20]:
    t = time.perf_counter()
    s = cross_val_score(logreg, X_train, y_train, cv=StratifiedKFold(k, shuffle=True, random_state=42))
    print(f"{k:>2} folds: {s.mean():.3f} ± {s.std():.3f}   ({time.perf_counter() - t:.2f} s)")

 3 folds: 0.683 ± 0.021   (0.01 s)
 5 folds: 0.685 ± 0.026   (0.01 s)
10 folds: 0.684 ± 0.031   (0.03 s)
20 folds: 0.682 ± 0.048   (0.05 s)


The mean barely moves (0.682 to 0.685). The standard deviation grows with more folds because each test fold is smaller and so noisier, and the time grows linearly with the number of models. Five or ten folds is the usual compromise.

## Exercise 4

In [6]:
from sklearn.tree import DecisionTreeClassifier
skf = StratifiedKFold(5, shuffle=True, random_state=42)
cv_means = {d: cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=42), X_train, y_train, cv=skf).mean()
            for d in range(1, 11)}
for d, m in cv_means.items():
    print(f"max_depth={d:>2}: {m:.3f}")
best = max(cv_means, key=cv_means.get)
tree = DecisionTreeClassifier(max_depth=best, random_state=42).fit(X_train, y_train)
print(f"\nchosen depth {best}; test accuracy {tree.score(X_test, y_test):.3f}")

max_depth= 1: 0.642
max_depth= 2: 0.659
max_depth= 3: 0.663
max_depth= 4: 0.669
max_depth= 5: 0.674
max_depth= 6: 0.649
max_depth= 7: 0.648
max_depth= 8: 0.642
max_depth= 9: 0.647
max_depth=10: 0.631

chosen depth 5; test accuracy 0.649


CV chooses depth 5 (0.674). Deeper trees score worse in CV: they start memorising the training folds (lesson 04.9). The single test score of 0.649 is lower than the CV estimate; with 830 test customers that gap is within normal split-to-split variation, and it is still the number you report.